# Captioner Benchmark - pick the VLM for Phase-2 scene-graph

We compare per-object captioning quality across local VLMs on a fixed set of YOLOe+SAM3 crops drawn from both AI2-THOR sim frames and Robo2VLM real-photo frames.

**Candidates (local Ollama):**
- `qwen3.5:9b` (Qwen-style VLM, 9.7B, vision capable)
- `qwen3.5:27b` (larger Qwen, 27B)
- `gemma4:26b` (Google Gemma multimodal, 25.8B)
- `gemma4:31b` (larger Gemma, ~30B)
- `qwen3-vl:8b`
- `qwen3-vl:30b`

**Scoring**:
- Latency per crop (median + p95)
- LLM-as-judge with GPT-5.4: given (image_crop, candidate captions list), pick the best
- Spot-check display so you can see captions side-by-side

**Excluded**: VLAs (OpenVLA, SmolVLA, StarVLA, RoboFlamingo) output discrete robot-action tokens, not natural language. They produce nonsense for this task.


## 0. Setup

In [ ]:
import sys, os, time, json, base64, io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import ollama

# Requires the package to be installed: pip install -e ".[analysis]" from the repo root.
from reflect.perception.open_world.detector import detect
from reflect.perception.open_world import filters as _filt
from reflect.perception.open_world import segmentor as _seg
from reflect.perception.open_world.captioner import caption, _crop_with_mask
from reflect.llm.prompter import PortkeyLLMPrompter
from reflect.perception.open_world import captioner as _capmod

from reflect.core.paths import analysis_experiment_dir

OUT_DIR = analysis_experiment_dir("vlm_capability_measurement")
OUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110})
print("ok")


## 1. Test set - 10 sim + 10 Robo2VLM frames

Caches the YOLOe+SAM3 crops to disk so we don't re-segment on every benchmark run.


In [ ]:
from reflect.core.paths import sim_data_root, analysis_cache_dir

SIM_ROOT = sim_data_root()  # honors REFLECT_DATA_ROOT
SIM_FRAMES = [
    ("boilWater", "boilWater-1", 5),  ("boilWater", "boilWater-1", 35),
    ("cookEgg", "cookEgg-1", 8),      ("cookEgg", "cookEgg-1", 33),
    ("toastBread", "toastBread-1", 6),("toastBread", "toastBread-1", 42),
    ("boilWater", "boilWater-1", 15), ("boilWater", "boilWater-1", 25),
    ("cookEgg", "cookEgg-1", 18),     ("toastBread", "toastBread-1", 22),
]

from datasets import load_from_disk
ROBO_DS = load_from_disk(str(analysis_cache_dir("robo2vlm_cache") / "sampled_subset"))  # honors REFLECT_OUTPUT_ROOT
ROBO_INDICES = [0, 3, 50, 100, 150, 200, 300, 500, 800, 1000]   # spread across the subset

CROPS_CACHE = OUT_DIR / "crops.pkl"

def get_test_crops(rebuild=False):
    if CROPS_CACHE.exists() and not rebuild:
        import pickle
        with open(CROPS_CACHE, "rb") as f:
            return pickle.load(f)

    items = []
    # Sim
    for task, ep, step in SIM_FRAMES:
        p = SIM_ROOT / task / ep / "ego_img" / f"img_step_{step}.png"
        rgb = np.array(Image.open(p).convert("RGB"))
        dets = detect(rgb, conf=0.10, max_detections=10)
        dets = _filt.filter_detections(dets, rgb.shape)
        dets = _filt.class_nms(dets, iou_thresh=0.4)
        dets = _filt.mask_subtract_contained(dets)
        masks = _seg.refine_masks(rgb, dets)
        for d, m in zip(dets, masks):
            if int(m.sum()) < 200:
                continue
            items.append(dict(
                source="sim", scene=f"{ep}#{step}",
                rgb=rgb, mask=m, bbox=d.bbox_xyxy,
                yoloe_label=d.class_name, yoloe_conf=d.confidence,
            ))
    # Real
    for idx in ROBO_INDICES:
        item = ROBO_DS[idx]
        rgb = np.array(item["image"].convert("RGB"))
        dets = detect(rgb, conf=0.10, max_detections=10)
        dets = _filt.filter_detections(dets, rgb.shape)
        dets = _filt.class_nms(dets, iou_thresh=0.4)
        dets = _filt.mask_subtract_contained(dets)
        masks = _seg.refine_masks(rgb, dets)
        for d, m in zip(dets, masks):
            if int(m.sum()) < 200:
                continue
            items.append(dict(
                source="real", scene=item["id"],
                rgb=rgb, mask=m, bbox=d.bbox_xyxy,
                yoloe_label=d.class_name, yoloe_conf=d.confidence,
            ))
    import pickle
    with open(CROPS_CACHE, "wb") as f:
        pickle.dump(items, f)
    return items

t0 = time.time()
crops = get_test_crops()
print(f"crops: {len(crops)} ({sum(1 for c in crops if c['source']=='sim')} sim, {sum(1 for c in crops if c['source']=='real')} real)  in {time.time()-t0:.1f}s")


## 2. Run candidate captioners over every crop

In [ ]:
# Pick the candidates you want to compare. Comment out heavyweights to save time.
CANDIDATES = [
    "qwen3.5:9b",        # Ollama local
    "qwen3.5:27b",       # Ollama local, larger
    "gemma4:26b",        # Ollama local
    "gemma4:31b",        # Ollama local, larger
    "qwen3-vl:8b",       # Ollama local
    "qwen3-vl:30b",      # Ollama local, larger
]

CAPTIONS_CACHE = OUT_DIR / "captions.json"
CAPTIONS_CACHE_VERSION = "mask-none-v2"

KNOWN_BACKENDS = _capmod.list_backends()
OLLAMA_BACKENDS = set(KNOWN_BACKENDS.get("ollama", []))
HF_BACKENDS = set(KNOWN_BACKENDS.get("hf", []))


def backend_kind(name: str) -> str:
    if name in HF_BACKENDS:
        return "hf"
    if name in OLLAMA_BACKENDS or name.startswith(("qwen3", "gemma")):
        return "ollama"
    return "other"


def ensure_ollama_models(candidates, auto_pull=True):
    installed = {m.model for m in ollama.list().models}
    needed = [b for b in candidates if backend_kind(b) == "ollama"]
    missing = [b for b in needed if b not in installed]

    if not missing:
        print("All Ollama candidate models are installed.")
        return []

    print("Missing Ollama models:", ", ".join(missing))
    if auto_pull:
        for model_name in missing:
            print(f"Pulling {model_name} ...")
            ollama.pull(model_name)
        print("Finished pulling missing models.")
    else:
        print("Set auto_pull=True below to download missing models before benchmarking.")
    return missing


def gpu_mem_used_mib():
    import subprocess
    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=memory.used",
                "--format=csv,noheader,nounits",
            ],
            text=True,
        ).strip()
        return int(out.splitlines()[0].strip())
    except Exception:
        return None


def stop_ollama_all(verbose=True):
    active_models = ollama.ps()
    models = active_models.get("models", [])

    if not models:
        if verbose:
            print("No active Ollama models to stop.")
        return

    for model in models:
        model_name = model["name"]
        ollama.generate(model_name, keep_alive=0)

    if verbose:
        print(f"Stopped {len(models)} Ollama model(s).")


def release_hf_cuda(verbose=True):
    import gc

    # Drop Python refs and clear torch allocator cache.
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

    # Clear cached HF model handles in captioner module.
    for fn_name in ("_load_hf_qwen_vl", "_load_hf_internvl"):
        fn = getattr(_capmod, fn_name, None)
        if fn and hasattr(fn, "cache_clear"):
            fn.cache_clear()

    if verbose:
        print("Released HF/PyTorch CUDA cache.")


def run_all(rebuild=False, checkpoint_every=25, log_vram=True):
    caption_mask_mode = "none"
    missing = ensure_ollama_models(CANDIDATES, auto_pull=True)
    if missing:
        raise RuntimeError("Install missing Ollama models or set auto_pull=True in ensure_ollama_models().")

    if CAPTIONS_CACHE.exists() and not rebuild:
        with open(CAPTIONS_CACHE) as f:
            data = json.load(f)
        if any(row.get("cache_version") != CAPTIONS_CACHE_VERSION for row in data):
            print("invalidated cached captions from the previous captioning config")
            data = []
        else:
            cleaned = []
            invalidated = 0
            for row in data:
                text = (row.get("text") or "").strip().lower()
                if row.get("backend", "").startswith("qwen3-vl") and text in {"<think>", "think", "thinking", "</think>"}:
                    invalidated += 1
                    continue
                cleaned.append(row)
            data = cleaned
            if invalidated:
                print(f"invalidated {invalidated} cached Qwen-VL think-only rows")
    else:
        data = []

    done = {(d["crop_idx"], d["backend"]) for d in data}
    total_todo = sum(
        1 for b in CANDIDATES for i in range(len(crops))
        if (i, b) not in done
    )
    print(f"to run: {total_todo}  (already cached: {len(done)})")

    written = 0
    prev_kind = None

    for backend in CANDIDATES:
        kind = backend_kind(backend)
        pending = [i for i in range(len(crops)) if (i, backend) not in done]
        if not pending:
            print(f"[skip] {backend}: nothing pending")
            continue

        # Cleanup when switching runtime families to avoid dual residency.
        if prev_kind and kind != prev_kind:
            if prev_kind == "ollama":
                stop_ollama_all(verbose=False)
            if prev_kind == "hf":
                release_hf_cuda(verbose=False)

        if kind == "ollama":
            stop_ollama_all(verbose=False)

        vram_before = gpu_mem_used_mib() if log_vram else None
        print(f"\n[backend] {backend} ({kind}): {len(pending)} crops")
        if vram_before is not None:
            print(f"  VRAM before: {vram_before} MiB")

        for i in pending:
            c = crops[i]
            try:
                r = caption(c["rgb"], c["mask"], c["bbox"], backend=backend, mask_mode=caption_mask_mode)
                data.append(dict(
                    crop_idx=i,
                    backend=backend,
                    text=r.text,
                    latency=r.latency_s,
                    source=c["source"],
                    yoloe_label=c["yoloe_label"],
                    cache_version=CAPTIONS_CACHE_VERSION,
                    caption_mask_mode=caption_mask_mode,
                ))
            except Exception as e:
                print(f"  [ERR] crop={i} backend={backend}: {type(e).__name__}: {str(e)[:120]}")
                data.append(dict(
                    crop_idx=i,
                    backend=backend,
                    text=f"<error: {type(e).__name__}>",
                    latency=None,
                    source=c["source"],
                    yoloe_label=c["yoloe_label"],
                    cache_version=CAPTIONS_CACHE_VERSION,
                    caption_mask_mode=caption_mask_mode,
                ))

            written += 1
            if written % checkpoint_every == 0:
                with open(CAPTIONS_CACHE, "w") as f:
                    json.dump(data, f, indent=1)

        # Release memory at backend boundary.
        if kind == "ollama":
            stop_ollama_all(verbose=False)
        elif kind == "hf":
            release_hf_cuda(verbose=False)

        vram_after = gpu_mem_used_mib() if log_vram else None
        if vram_after is not None:
            print(f"  VRAM after:  {vram_after} MiB")

        prev_kind = kind

        with open(CAPTIONS_CACHE, "w") as f:
            json.dump(data, f, indent=1)

    with open(CAPTIONS_CACHE, "w") as f:
        json.dump(data, f, indent=1)
    return data

caps = run_all()
df = pd.DataFrame(caps)
print(f"\nlatency summary (s):")
print(df.groupby("backend")["latency"].agg(["count", "median", lambda x: x.dropna().quantile(0.95) if len(x.dropna()) else None]).rename(columns={"<lambda_0>": "p95"}))

## 3. Side-by-side display

In [ ]:
def show_crop_grid(crop_items, captions_df, n=10, source=None):
    items = [
        (i, c) for i, c in enumerate(crop_items)
        if (source is None or c["source"] == source)
    ][:n]

    if not items:
        return

    _, axes = plt.subplots(
        len(items), 2,
        figsize=(13, 2.6 * len(items)),
        gridspec_kw={"width_ratios": [1, 2]}
    )

    if len(items) == 1:
        axes = axes[None, :]

    for r, (idx, c) in enumerate(items):
        crop_img = _crop_with_mask(c["rgb"], c["mask"], c["bbox"])

        axes[r, 0].imshow(crop_img)
        axes[r, 0].axis("off")
        axes[r, 0].set_title(f"YOLOe: {c['yoloe_label']}", fontsize=8)

        text_panel = axes[r, 1]
        text_panel.axis("off")

        rows = captions_df[captions_df["crop_idx"] == idx]

        y = 0.95
        for _, row in rows.iterrows():
            text_panel.text(
                0.0, y,
                f"{row['backend']:<14s}: {row['text']}",
                fontsize=9,
                family="monospace",
                va="top",
                transform=text_panel.transAxes
            )
            y -= 0.16

    plt.suptitle(f"Source: {source}" if source else "All", fontweight="bold")
    plt.tight_layout()
    plt.show()

show_crop_grid(crops, df, n=10, source="sim")
show_crop_grid(crops, df, n=10, source="real")


## 4. LLM-as-judge with GPT-5.4

For each crop, ask GPT-5.4 to rank the candidate captions.


In [ ]:
PORTKEY_API_KEY = os.environ["PORTKEY_API_KEY"]  # set in your shell or .env (see .env.example)
judge = PortkeyLLMPrompter(portkey_api_key=PORTKEY_API_KEY, model="gpt-5.4", reasoning_effort="none")

def crop_to_b64(c):
    img = _crop_with_mask(c["rgb"], c["mask"], c["bbox"])
    buf = io.BytesIO(); img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

JUDGE_PROMPT = (
    "You will see an image of a single object (most other content is darkened). "
    "Below are several candidate one-line captions of that object from different VLMs. "
    "Pick the SINGLE caption that most accurately and specifically identifies the object. "
    "Tie-break by specificity (e.g., 'silver faucet' beats 'metal object'). "
    "Reply with ONLY the backend name (e.g., 'qwen3.5:9b') of the winning caption - "
    "nothing else, no quotes, no punctuation. "
    "If ALL captions are equally bad/wrong, reply 'NONE'."
)

JUDGE_CACHE = OUT_DIR / "judge.json"

def judge_all(rebuild=False):
    if JUDGE_CACHE.exists() and not rebuild:
        with open(JUDGE_CACHE) as f: out = json.load(f)
    else:
        out = []
    done = {d["crop_idx"] for d in out}
    rows_by_crop = df.groupby("crop_idx")
    for crop_idx, rows in rows_by_crop:
        if crop_idx in done: continue
        c = crops[crop_idx]
        b64 = crop_to_b64(c)
        candidates_text = "\n".join(f"- {r['backend']}: {r['text']}" for _, r in rows.iterrows())
        msg = [{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            {"type": "text", "text": JUDGE_PROMPT + "\n\nCandidates:\n" + candidates_text},
        ]}]
        try:
            resp = judge._client.chat.completions.create(
                model="gpt-5.4", messages=msg, max_completion_tokens=20,
            )
            winner = resp.choices[0].message.content.strip()
        except Exception as e:
            winner = f"<err:{type(e).__name__}>"
        out.append(dict(crop_idx=int(crop_idx), source=c["source"],
                         yoloe_label=c["yoloe_label"], winner=winner))
        if len(out) % 10 == 0:
            with open(JUDGE_CACHE, "w") as f: json.dump(out, f, indent=1)
            print(f"  judged {len(out)}/{len(crops)}")
    with open(JUDGE_CACHE, "w") as f: json.dump(out, f, indent=1)
    return out

judgements = judge_all()
jdf = pd.DataFrame(judgements)
print(f"judged {len(jdf)} crops")
print()
print("Winner by source:")
print(pd.crosstab(jdf["source"], jdf["winner"]))


## 5. Final summary - pick the winner

In [ ]:
# Win-rate per backend (overall + per source)
def winrate(d):
    counts = d["winner"].value_counts(normalize=True)
    return counts

print("=== OVERALL win rate ===")
print(winrate(jdf).to_string())
print()
print("=== SIM only ===")
print(winrate(jdf[jdf["source"] == "sim"]).to_string())
print()
print("=== REAL only ===")
print(winrate(jdf[jdf["source"] == "real"]).to_string())
print()
print("=== Latency (seconds) ===")
lat = df.groupby("backend")["latency"].agg(["median",
        lambda x: x.dropna().quantile(0.95) if len(x.dropna()) else None]).rename(columns={"<lambda_0>": "p95"})
print(lat.to_string())

# Recommend
print("\n--- Recommendation ---")
overall_winner = jdf["winner"].value_counts().idxmax()
print(f"Most-picked overall: {overall_winner}")
